# Mini Project: Books Dataset via Google Books API

Dataset diambil dari **Google Books API**, dibungkus dalam sebuah class, lalu dibersihkan (missing value, duplikat, tipe data) sebelum disimpan sebagai `books_dataset.csv`.

## 1. Import Library & Load API Key

API key disimpan di file `.env` (tidak ditulis langsung di notebook).

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()


## 2. Class `GoogleBooksAPI`

- **Atribut:** `api_key` (dari `.env`), `base_url` (endpoint Google Books API)
- **Method:** `get_books()` memanggil API dan mengembalikan data dalam bentuk `list of dict`, dibungkus `try-except` supaya program tidak berhenti total kalau koneksi bermasalah

In [ ]:
class GoogleBooksAPI:
    def __init__(self):
        self.api_key = os.getenv("GOOGLE_BOOKS_API_KEY")
        self.base_url = "https://www.googleapis.com/books/v1/volumes"

    def get_books(self, query, max_results=40, start_index=0):
        """Memanggil API dan mengembalikan data buku dalam bentuk list of dict"""
        params = {
            "q": query,
            "key": self.api_key,
            "maxResults": max_results,
            "startIndex": start_index
        }

        try:
            response = requests.get(self.base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Gagal mengambil data untuk query='{query}': {e}")
            return []

        books = []
        for item in data.get("items", []):
            info = item.get("volumeInfo", {})
            books.append({
                "title": info.get("title"),
                "authors": ", ".join(info.get("authors", [])) if info.get("authors") else None,
                "publisher": info.get("publisher"),
                "published_date": info.get("publishedDate"),
                "page_count": info.get("pageCount"),
                "categories": ", ".join(info.get("categories", [])) if info.get("categories") else None,
                "average_rating": info.get("averageRating"),
                "language": info.get("language"),
            })
        return books


## 3. Pengambilan Data

Karena satu kali panggilan API dibatasi maksimal ±20 hasil, API dipanggil berulang dengan beberapa kata kunci berbeda, lalu hasilnya digabung.

In [ ]:
api = GoogleBooksAPI()

keywords = ["fiction", "history", "science", "biography", "technology", "fantasy", "romance", "mystery"]

all_books = []
for kw in keywords:
    result = api.get_books(kw, max_results=40)
    print(f"Query '{kw}' -> {len(result)} buku")
    all_books.extend(result)

df = pd.DataFrame(all_books)
print(f"\nTotal data sebelum dibersihkan: {len(df)}")
df.head()


## 4. Function Pembersih Data

Ada 3 function pembersih: menangani nilai kosong, membuang data kembar, dan memastikan tipe data.

### 4.1 Menangani Nilai Kosong

| Kolom | Keputusan | Alasan |
|---|---|---|
| `title` | Baris dibuang jika kosong | Judul adalah identitas utama buku |
| `authors` | Diisi "Tidak diketahui" | Identitas unik per buku, tidak valid ditebak pakai modus |
| `publisher` | Diisi "Tidak diketahui" | Sama seperti authors, spesifik per buku |
| `categories` | Diisi "Tidak diketahui" | Spesifik per buku, tidak bisa digeneralisasi |
| `language` | Diisi modus | Ada pola dominan bahasa dalam dataset |
| `page_count` | Diisi rata-rata (mean) | Kolom angka, estimasi statistik wajar |
| `average_rating` | Diisi rata-rata (mean) | Kolom angka |
| `published_date` | Diisi median | Median lebih tahan outlier dibanding mean untuk tanggal |

In [ ]:
def handle_missing_values(df):
    """
    Menangani nilai kosong di tiap kolom (lihat penjelasan alasan di sel markdown di atas).
    """
    df = df.dropna(subset=["title"])

    df["authors"] = df["authors"].fillna("Tidak diketahui")
    df["publisher"] = df["publisher"].fillna("Tidak diketahui")
    df["categories"] = df["categories"].fillna("Tidak diketahui")

    if not df["language"].mode().empty:
        df["language"] = df["language"].fillna(df["language"].mode()[0])
    else:
        df["language"] = df["language"].fillna("unknown")

    df["page_count"] = pd.to_numeric(df["page_count"], errors="coerce")
    df["page_count"] = df["page_count"].fillna(df["page_count"].mean())

    df["average_rating"] = pd.to_numeric(df["average_rating"], errors="coerce")
    df["average_rating"] = df["average_rating"].fillna(df["average_rating"].mean())

    df["published_date"] = pd.to_datetime(df["published_date"], errors="coerce")
    if df["published_date"].notna().sum() > 0:
        median_date = df["published_date"].median()
        df["published_date"] = df["published_date"].fillna(median_date)

    return df


### 4.2 Membuang Data Kembar

Buku yang sama bisa muncul dua kali karena kata kunci pencarian berbeda. Dianggap sama kalau `title` dan `authors` sama persis.

In [ ]:
def remove_duplicates(df):
    """Membuang baris yang sebenarnya sama (title + authors sama)."""
    before = len(df)
    df = df.drop_duplicates(subset=["title", "authors"], keep="first")
    after = len(df)
    print(f"Duplikat dibuang: {before - after} baris")
    return df


### 4.3 Memastikan Tipe Data

`page_count` → integer, `average_rating` → float, `published_date` → datetime.

In [ ]:
def fix_data_types(df):
    """Memastikan tipe data tiap kolom sesuai (angka jadi angka, tanggal jadi tanggal)."""
    df["page_count"] = pd.to_numeric(df["page_count"], errors="coerce").fillna(0).astype(int)
    df["average_rating"] = pd.to_numeric(df["average_rating"], errors="coerce").fillna(0).astype(float)
    df["published_date"] = pd.to_datetime(df["published_date"], errors="coerce")
    return df


## 5. Menjalankan Proses Pembersihan

In [ ]:
df = handle_missing_values(df)
df = remove_duplicates(df)
df = fix_data_types(df)

print(f"Total data setelah dibersihkan: {len(df)}")
print("\nInfo tipe data tiap kolom:")
print(df.dtypes)


## 6. Cuplikan Dataset Hasil Akhir

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

## 7. Simpan Dataset

In [ ]:
df.to_csv("books_dataset.csv", index=False)
print("Data disimpan ke books_dataset.csv")
